# 01 — Dataset Collection & Research
**Project:** Uber Surge Multiplier Prediction (Regression)

**Source:** Kaggle — *Uber and Lyft Dataset Boston, MA* (`brllrb/uber-and-lyft-dataset-boston-ma`).
The original file `rideshare_kaggle.csv` is stored unchanged as `data/raw/uber_raw.csv`.

Goal of this notebook: inspect the raw data, prove that a **real, recorded surge multiplier** exists,
and document what is available / missing. No cleaning and no modelling here.

In [1]:
# --- Project setup (works locally and on Colab after cloning the repo) ---
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: D:\ML


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/uber_raw.csv")

print("Dataset Shape:")
print(df.shape)

print("First 5 Rows:")
display(df.head())

Dataset Shape:
(693071, 57)
First 5 Rows:


,id,timestamp,hour,day,month,datetime,timezone,source,destination,cab_type,...,precipIntensityMax,uvIndexTime,temperatureMin,temperatureMinTime,temperatureMax,temperatureMaxTime,apparentTemperatureMin,apparentTemperatureMinTime,apparentTemperatureMax,apparentTemperatureMaxTime
0,424553bb-7174-41ea-aeb4-fe06d4f4b9d7,1.544953e+09,9,16,12,2018-12-16 09:30:07,America/New_York,Haymarket Square,North Station,Lyft,...,0.1276,1544979600,39.89,1545012000,43.68,1544968800,33.73,1545012000,38.07,1544958000
1,4bd23055-6827-41c6-b23b-3c491f24e74d,1.543284e+09,2,27,11,2018-11-27 02:00:23,America/New_York,Haymarket Square,North Station,Lyft,...,0.1300,1543251600,40.49,1543233600,47.30,1543251600,36.20,1543291200,43.92,1543251600
2,981a3613-77af-4620-a42a-0c0866077d1e,1.543367e+09,1,28,11,2018-11-28 01:00:22,America/New_York,Haymarket Square,North Station,Lyft,...,0.1064,1543338000,35.36,1543377600,47.55,1543320000,31.04,1543377600,44.12,1543320000
3,c2d88af2-d278-4bfd-a8d0-29ca77cc5512,1.543554e+09,4,30,11,2018-11-30 04:53:02,America/New_York,Haymarket Square,North Station,Lyft,...,0.0000,1543507200,34.67,1543550400,45.03,1543510800,30.30,1543550400,38.53,1543510800
4,e0126e1f-8ca9-4f2e-82b3-50505a09db9a,1.543463e+09,3,29,11,2018-11-29 03:49:20,America/New_York,Haymarket Square,North Station,Lyft,...,0.0001,1543420800,33.10,1543402800,42.18,1543420800,29.11,1543392000,35.75,1543420800


In [3]:
print("Dataset Information:")
df.info()

Dataset Information:


<class 'pandas.DataFrame'>
RangeIndex: 693071 entries, 0 to 693070
Data columns (total 57 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           693071 non-null  str    
 1   timestamp                    693071 non-null  float64
 2   hour                         693071 non-null  int64  
 3   day                          693071 non-null  int64  
 4   month                        693071 non-null  int64  
 5   datetime                     693071 non-null  str    
 6   timezone                     693071 non-null  str    
 7   source                       693071 non-null  str    
 8   destination                  693071 non-null  str    
 9   cab_type                     693071 non-null  str    
 10  product_id                   693071 non-null  str    
 11  name                         693071 non-null  str    
 12  price                        637976 non-null  float64
 13  distance  

In [4]:
print("Statistical Description:")
display(df.describe().T)

Statistical Description:


,count,mean,std,min,25%,50%,75%,max
timestamp,693071.0,1.544046e+09,689192.492586,1.543204e+09,1.543444e+09,1.543737e+09,1.544828e+09,1.545161e+09
hour,693071.0,1.161914e+01,6.948114,0.000000e+00,6.000000e+00,1.200000e+01,1.800000e+01,2.300000e+01
day,693071.0,1.779436e+01,9.982286,1.000000e+00,1.300000e+01,1.700000e+01,2.800000e+01,3.000000e+01
month,693071.0,1.158668e+01,0.492429,1.100000e+01,1.100000e+01,1.200000e+01,1.200000e+01,1.200000e+01
price,637976.0,1.654513e+01,9.324359,2.500000e+00,9.000000e+00,1.350000e+01,2.250000e+01,9.750000e+01
distance,693071.0,2.189430e+00,1.138937,2.000000e-02,1.280000e+00,2.160000e+00,2.920000e+00,7.860000e+00
surge_multiplier,693071.0,1.013870e+00,0.091641,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00
latitude,693071.0,4.233817e+01,0.047840,4.221480e+01,4.235030e+01,4.235190e+01,4.236470e+01,4.236610e+01
longitude,693071.0,-7.106615e+01,0.020302,-7.110540e+01,-7.108100e+01,-7.106310e+01,-7.105420e+01,-7.103300e+01
temperature,693071.0,3.958439e+01,6.726084,1.891000e+01,3.645000e+01,4.049000e+01,4.358000e+01,5.722000e+01


## Target check — `surge_multiplier`
The column is recorded by the data collector from the ride-hailing APIs (it is **not** derived from price).

In [5]:
target = "surge_multiplier"

print(df[target].describe())

print("Missing Targets:")
print(df[target].isnull().sum())

print("Unique Surge Values:")
print(df[target].nunique())

print(df[target].value_counts().head(20))

count    693071.000000
mean          1.013870
std           0.091641
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           3.000000
Name: surge_multiplier, dtype: float64
Missing Targets:
0
Unique Surge Values:
7
surge_multiplier
1.00    672096
1.25     11085
1.50      5065
1.75      2420
2.00      2239
2.50       154
3.00        12
Name: count, dtype: int64


### Surge per company — key finding

In [6]:
display(pd.crosstab(df["cab_type"], df[target]))

surge_multiplier,1.00,1.25,1.50,1.75,2.00,2.50,3.00
cab_type,,,,,,,
Lyft,286433,11085,5065,2420,2239,154,12
Uber,385663,0,0,0,0,0,0


**Finding:** every Uber row has `surge_multiplier = 1.0` (385,663 rows) — the collector did not receive
Uber's surge value, so a constant 1.0 there is *not* a real observation.
All the real variation (1.0 → 3.0) comes from **Lyft** rows.

**Team decision:** the project uses the **Lyft rows only** (≈307k trips), where the surge multiplier is genuinely recorded.
The problem type stays the same (regression of a ride-hailing surge multiplier), which the project-ideas sheet allows
("similar dataset from the same category") — the TA must be informed. The filtering itself is done in notebook 02 (cleaning).

In [7]:
lyft = df[df["cab_type"] == "Lyft"]
print("Lyft rows:", len(lyft))
print("Share of Lyft rides with surge > 1:", round((lyft[target] > 1).mean() * 100, 2), "%")
display(pd.crosstab(lyft["name"], lyft[target]))

Lyft rows: 307408
Share of Lyft rides with surge > 1: 6.82 %


surge_multiplier,1.00,1.25,1.50,1.75,2.00,2.50,3.00
name,,,,,,,
Lux,47040,2217,1013,484,481,0,0
Lux Black,47040,2217,1013,484,481,0,0
Lux Black XL,47040,2217,1013,484,481,0,0
Lyft,47040,2217,1013,484,398,77,6
Lyft XL,47040,2217,1013,484,398,77,6
Shared,51233,0,0,0,0,0,0


Lyft's *Shared* product never surges; all other Lyft products share the same surge for the same request
(each API request returned one row per product).

## Initial quality check (no cleaning here — that's notebook 02)

In [8]:
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0])

print("Duplicate Rows:")
print(df.duplicated().sum())

print("Duplicate ids:")
print(df["id"].duplicated().sum())

Missing Values:
price    55095
dtype: int64
Duplicate Rows:


0
Duplicate ids:


0


In [9]:
print("Data Types:")
print(df.dtypes.value_counts())
print("Time range:", df["datetime"].min(), "->", df["datetime"].max())

Data Types:
float64    29
int64      17
str        11
Name: count, dtype: int64
Time range: 2018-11-26 03:40:46 -> 2018-12-18 19:15:10


## Availability vs. the project's wish-list

| Wanted column | Available as | Notes |
|---|---|---|
| Pickup Location | `source` (12 Boston areas) + `latitude`/`longitude` | area level, not exact address |
| Dropoff Location | `destination` | 12 areas |
| Trip Distance | `distance` (miles) | |
| Pickup Date / Time | `datetime`, `timestamp`, `hour`, `day`, `month` | Nov 26 – Dec 18, 2018 |
| Surge Multiplier | `surge_multiplier` | real for Lyft only |
| Weather | ~40 weather columns (`temperature`, `precipIntensity`, `humidity`, `short_summary`, …) | already joined by the dataset author per area & hour |
| Traffic | **not available** | not added — no reliable source to join |
| Available Drivers | **not available** | not added |
| Ride Requests | **not available** | not added |

Summary: rows / columns and the full column documentation are in `reports/data_dictionary.md` and `reports/dataset_sources.md`.